# Release16 Nonconvex Checkpoint BC Official Replay

## Governance

- Baselines: `soxlbacktest_qlib_walkforward_baseline_v1` as the qlib buy/sell reference, `release16_bull_reentry_participation_objective_replay_v1::bull65_hold3` as the best runnable official replay reference, and `release16_nonconvex_checkpoint_objective_score_handoff_v1` as the direct handoff source.
- Role: `diagnose`; no actor retention is allowed from this screen.
- Target windows: `wf_2020`, `wf_2022`, `wf_2023_2024`.
- Accepted trade-off: each selected fold candidate must target relative return `>= 30pp`, official warm-start MaxDD `< 20%`, sparse slice fit, and no drawdown drift versus the runnable reference.
- Changed layer: `model_selection`, `bc_warm_start`, `checkpoint_objective`, `official_replay`.
- External reference: qlib official daily replay and the repo-owned SOXLBacktest qlib buy/sell baseline.
- Hypothesis: fold-local state-head target contract can train a BC warm-start actor that preserves the SOXLBacktest buy/sell timing idea while reducing drawdown drift enough to be a checkpoint/reward target.
- Failure mode: global BC accuracy can pass while sparse release/rebound/downtrend slices fail, or official replay can miss the relative-return floor.
- Historical success case: `cap10_three_fold_bc_slice_screen_v1` showed BC slice screens can become checkpoint handoffs.
- Historical failure case: `release16_release_gate_bc_official_replay_v1` failed official replay gates after release-gate weighting.
- Why this is not duplicate: this card starts from the newer `release16_nonconvex_checkpoint_objective_score_handoff_v1` target contract, includes official warm-start replay for all required folds, and scores against both the best runnable model reference and SOXLBacktest qlib baseline.


In [1]:

from __future__ import annotations

import json
import math
import os
import random
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_conditioned_objective import daily_path_metrics, max_drawdown_interval_metrics
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCSampleWeightConfig,
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
    score_bc_walkforward_drift,
    summarize_bc_walkforward_drift_screen,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.core.metric_reconciliation import build_warm_start_replay_segment, slice_warm_start_daily_path
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, flatten_feature_frame, init_qlib
from src.soxl_quant.settings import LOOKBACK, QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'release16_nonconvex_checkpoint_bc_official_replay_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
DAILY_DIR = CARD / 'official_daily_paths'
DAILY_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_SCORE_CARD = ROOT / 'runtime/model_selection/card/release16_nonconvex_checkpoint_objective_score_handoff_v1'
READINESS_CARD = ROOT / 'runtime/model_selection/card/release16_nonconvex_checkpoint_objective_readiness_v1'
TARGET_CONTRACT_CARD = ROOT / 'runtime/model_selection/card/release16_failedfold_state_head_target_recipe_v1'
SOXL_BASELINE_CARD = ROOT / 'runtime/model_selection/card/soxlbacktest_qlib_walkforward_baseline_v1'
TARGETS_PATH = TARGET_CONTRACT_CARD / 'state_head_target_contract.csv'
RUNNABLE_REFERENCE_PATH = SOURCE_SCORE_CARD / 'best_runnable_window_terms.csv'
SOXL_BASELINE_METRICS_PATH = SOXL_BASELINE_CARD / 'candidate_metrics.csv'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260615
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 20.0
DRIFT_RELATIVE_FLOOR_PP = -5.0
DRIFT_MAX_DD_CEILING_PP = 2.0
PROFILE_NAME, CARD


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


('sparse_release_bc_ppo_screen_v1',
 PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_nonconvex_checkpoint_bc_official_replay_v1'))

In [2]:

WALK_FORWARD_WINDOWS = {
    'wf_2020': {
        'target_fold': 'wf_2020_train',
        'train': ('2010-03-11', '2019-12-31'),
        'valid': ('2020-01-01', '2020-12-31'),
        'test': ('2020-01-01', '2020-12-31'),
    },
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}

BC_TRAIN_CONFIGS = {
    'balanced_sparse': StateObjectiveBCTrainingConfig(steps=220, lr=0.001, batch_size=128, seed=SEED),
    'release_heavy': StateObjectiveBCTrainingConfig(steps=260, lr=0.001, batch_size=128, seed=SEED + 11),
    'risk_guarded': StateObjectiveBCTrainingConfig(steps=260, lr=0.0007, batch_size=128, seed=SEED + 23),
}
BC_WEIGHT_CONFIGS = {
    'balanced_sparse': StateObjectiveBCSampleWeightConfig(
        rebound_multiplier=5.0,
        downtrend_multiplier=3.0,
        release_gate_multiplier=12.0,
        ordinary_multiplier=1.0,
        objective_weight_power=0.25,
        class_balanced_beta=0.0,
        weight_floor=0.05,
        weight_cap=35.0,
        normalize_mean=True,
    ),
    'release_heavy': StateObjectiveBCSampleWeightConfig(
        rebound_multiplier=8.0,
        downtrend_multiplier=4.0,
        release_gate_multiplier=24.0,
        ordinary_multiplier=1.0,
        objective_weight_power=0.5,
        class_balanced_beta=0.0,
        weight_floor=0.05,
        weight_cap=50.0,
        normalize_mean=True,
    ),
    'risk_guarded': StateObjectiveBCSampleWeightConfig(
        rebound_multiplier=4.0,
        downtrend_multiplier=10.0,
        release_gate_multiplier=14.0,
        ordinary_multiplier=1.0,
        objective_weight_power=0.5,
        class_balanced_beta=0.0,
        weight_floor=0.05,
        weight_cap=50.0,
        normalize_mean=True,
    ),
}
SLICE_RULES = {
    'release_gate_state': {'min_samples': 20, 'min_accuracy': 0.60},
    'is_rebound_cluster': {'min_samples': 20, 'min_accuracy': 0.60},
    'is_downtrend_state': {'min_samples': 50, 'min_accuracy': 0.60},
    'drawdown_control_state': {'min_samples': 20, 'min_accuracy': 0.60},
    'failedfold_state_head_target_state': {'min_samples': 5, 'min_accuracy': 0.55},
    'is_high_participation_target': {'min_samples': 20, 'min_accuracy': 0.60},
}
SLICE_COLS = tuple(SLICE_RULES)


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return str(value.date())
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')


def set_seed(seed: int) -> None:
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))


def datetime_index_frame(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    if isinstance(out.index, pd.MultiIndex):
        if 'instrument' in out.index.names:
            out = out.droplevel('instrument')
        else:
            out = out.reset_index(level=list(range(1, out.index.nlevels)), drop=True)
    out.index = pd.DatetimeIndex(pd.to_datetime(out.index))
    return out[~out.index.duplicated(keep='last')].sort_index()


def datetime_index_series(series: pd.Series) -> pd.Series:
    out = series.copy()
    if isinstance(out.index, pd.MultiIndex):
        if 'instrument' in out.index.names:
            out = out.droplevel('instrument')
        else:
            out.index = out.index.get_level_values(0)
    out.index = pd.DatetimeIndex(pd.to_datetime(out.index))
    return out[~out.index.duplicated(keep='last')].sort_index()


def slice_gate_from_metrics(slice_metrics: pd.DataFrame) -> tuple[bool, dict[str, Any], list[str]]:
    details: dict[str, Any] = {}
    failed: list[str] = []
    indexed = slice_metrics.set_index('slice') if not slice_metrics.empty else pd.DataFrame()
    for name, rule in SLICE_RULES.items():
        if name not in indexed.index:
            details[name] = {'samples': 0, 'accuracy': None, 'min_samples': rule['min_samples'], 'min_accuracy': rule['min_accuracy']}
            failed.append(name)
            continue
        row = indexed.loc[name]
        samples = int(row.get('samples', 0))
        accuracy = float(row.get('accuracy', float('nan')))
        ok = samples >= int(rule['min_samples']) and math.isfinite(accuracy) and accuracy >= float(rule['min_accuracy'])
        details[name] = {'samples': samples, 'accuracy': accuracy, 'min_samples': rule['min_samples'], 'min_accuracy': rule['min_accuracy'], 'gate': bool(ok)}
        if not ok:
            failed.append(name)
    return (len(failed) == 0), details, failed


def candidate_loss(row: dict[str, Any]) -> float:
    relative_gap = max(0.0, TARGET_RELATIVE_RETURN - float(row.get('relative_return', float('-inf'))))
    dd_gap = max(0.0, float(row.get('max_dd', float('inf'))) - TARGET_MAX_DD)
    slice_penalty = 12.0 * int(not bool(row.get('slice_gate')))
    drift_penalty = max(0.0, DRIFT_RELATIVE_FLOOR_PP - float(row.get('relative_delta_vs_runnable', float('-inf'))))
    drift_penalty += max(0.0, float(row.get('max_dd_delta_vs_runnable', float('inf'))) - DRIFT_MAX_DD_CEILING_PP) * 2.0
    return float(relative_gap + dd_gap * 4.0 + slice_penalty + drift_penalty)


def baseline_metrics_by_fold(frame: pd.DataFrame, candidate: str | None = None) -> dict[str, dict[str, float]]:
    out: dict[str, dict[str, float]] = {}
    work = frame.copy()
    if candidate and 'candidate' in work.columns:
        work = work[work['candidate'].astype(str).eq(candidate)]
    for _, row in work.iterrows():
        fold = str(row.get('window') or row.get('fold') or '')
        if not fold.startswith('wf_'):
            continue
        rel = row.get('relative_return')
        try:
            rel_missing = pd.isna(rel)
        except (TypeError, ValueError):
            rel_missing = False
        if rel_missing and {'rulebase_return_pct', 'soxl_buyhold_return_pct'}.issubset(work.columns):
            rel = float(row['rulebase_return_pct']) - float(row['soxl_buyhold_return_pct'])
        max_dd = row.get('max_dd')
        try:
            dd_missing = pd.isna(max_dd)
        except (TypeError, ValueError):
            dd_missing = False
        if dd_missing and 'rulebase_max_drawdown_pct' in work.columns:
            max_dd = row['rulebase_max_drawdown_pct']
        avg_pos = row.get('avg_pos')
        try:
            pos_missing = pd.isna(avg_pos)
        except (TypeError, ValueError):
            pos_missing = False
        if pos_missing and 'average_position_pct' in work.columns:
            avg_pos = float(row['average_position_pct']) / 100.0
        out[fold] = {
            'relative_return': float(rel),
            'max_dd': float(max_dd),
            'avg_pos': float(avg_pos),
        }
    return out

print('root', ROOT)
print('card', CARD)


root /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
card /Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/runtime/model_selection/card/release16_nonconvex_checkpoint_bc_official_replay_v1


In [3]:

profile = load_profile(PROFILE_NAME)
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)
targets = pd.read_csv(TARGETS_PATH, parse_dates=['datetime'])
runnable_reference = pd.read_csv(RUNNABLE_REFERENCE_PATH)
soxl_baseline_metrics = pd.read_csv(SOXL_BASELINE_METRICS_PATH)
source_summary = json.loads((SOURCE_SCORE_CARD / 'summary.json').read_text(encoding='utf-8'))
readiness_summary = json.loads((READINESS_CARD / 'summary.json').read_text(encoding='utf-8'))

runnable_by_fold = baseline_metrics_by_fold(runnable_reference, candidate='bull65_hold3')
soxl_legacy_by_fold = baseline_metrics_by_fold(soxl_baseline_metrics, candidate='legacy_default')
soxl_risk_floor_by_fold = baseline_metrics_by_fold(soxl_baseline_metrics, candidate='wf_pass_ytd18_15_15_02')

init_qlib(QLIB_DATA_PATH)

coverage_rows: list[dict[str, Any]] = []
fold_cache: dict[str, dict[str, Any]] = {}
for fold, spec in WALK_FORWARD_WINDOWS.items():
    target_fold = str(spec['target_fold'])
    train_start, train_end = map(pd.Timestamp, spec['train'])
    test_start, test_end = map(pd.Timestamp, spec['test'])
    fold_targets = targets[targets['fold'].astype(str).eq(target_fold)].copy()
    train_targets = fold_targets[(fold_targets['datetime'] >= train_start) & (fold_targets['datetime'] <= train_end)].copy()
    overlap = train_targets[(train_targets['datetime'] >= test_start) & (train_targets['datetime'] <= test_end)]
    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    feature_full = datetime_index_frame(flatten_feature_frame(dataset.handler.feature_frame))
    price_full = datetime_index_series(dataset.handler.price_frame['price'])
    train_frame = feature_full.loc[spec['train'][0]:spec['train'][1]].copy()
    warm_seg = build_warm_start_replay_segment(feature_full.index, eval_seg=spec['test'], lookback=LOOKBACK)
    fold_cache[fold] = {
        'dataset': dataset,
        'feature_full': feature_full,
        'price_full': price_full,
        'train_frame': train_frame,
        'warm_seg': warm_seg,
        'targets': train_targets,
    }
    coverage_rows.append({
        'fold': fold,
        'target_fold': target_fold,
        'target_rows': int(len(train_targets)),
        'target_start': str(train_targets['datetime'].min().date()) if len(train_targets) else '',
        'target_end': str(train_targets['datetime'].max().date()) if len(train_targets) else '',
        'train_start': spec['train'][0],
        'train_end': spec['train'][1],
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
        'feature_start': str(feature_full.index.min().date()),
        'feature_end': str(feature_full.index.max().date()),
        'train_frame_rows': int(len(train_frame)),
        'warm_start': warm_seg[0],
        'warm_end': warm_seg[1],
        'eval_overlap_rows': int(len(overlap)),
        'fold_local_gate': bool(len(train_targets) > 0 and len(overlap) == 0),
    })

coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / 'target_coverage.csv', index=False)
display(coverage)


[20293:MainThread](2026-06-15 09:21:53,110) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[20293:MainThread](2026-06-15 09:21:53,553) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[20293:MainThread](2026-06-15 09:21:53,553) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix/data/qlib/soxl')}


[20293:MainThread](2026-06-15 09:21:54,556) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[20293:MainThread](2026-06-15 09:21:54,557) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


...

[20293:MainThread](2026-06-15 09:21:55,666) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


[20293:MainThread](2026-06-15 09:21:56,941) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | Loading data Done


[20293:MainThread](2026-06-15 09:21:56,941) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[20293:MainThread](2026-06-15 09:21:56,941) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


,fold,target_fold,target_rows,target_start,target_end,train_start,train_end,test_start,test_end,feature_start,feature_end,train_frame_rows,warm_start,warm_end,eval_overlap_rows,fold_local_gate
0,wf_2020,wf_2020_train,2470,2010-03-11,2019-12-31,2010-03-11,2019-12-31,2020-01-01,2020-12-31,2011-03-11,2020-12-31,2217,2019-10-07,2020-12-31,0,True
1,wf_2022,wf_2022_train,253,2020-01-02,2020-12-31,2010-03-11,2020-12-31,2022-01-01,2022-12-31,2011-03-11,2022-12-30,2470,2021-10-07,2022-12-30,0,True
2,wf_2023_2024,wf_2023_2024_train,756,2020-01-02,2022-12-30,2010-03-11,2022-12-31,2024-01-01,2024-12-31,2011-03-11,2024-12-31,2973,2023-10-05,2024-12-31,0,True


In [4]:

candidate_rows: list[dict[str, Any]] = []
fit_rows: list[dict[str, Any]] = []
slice_frames: list[pd.DataFrame] = []
action_frames: list[pd.DataFrame] = []
daily_manifest: list[dict[str, Any]] = []
drawdown_rows: list[dict[str, Any]] = []
soxl_baseline_rows: list[dict[str, Any]] = []

for fold, spec in WALK_FORWARD_WINDOWS.items():
    cache = fold_cache[fold]
    fold_targets = cache['targets']
    if fold_targets.empty:
        candidate_rows.append({'fold': fold, 'candidate': 'none', 'status': 'skipped_no_targets'})
        continue
    for i, (candidate, weight_config) in enumerate(BC_WEIGHT_CONFIGS.items()):
        print(f'[fold:{fold}] candidate={candidate} targets={len(fold_targets)} warm={cache["warm_seg"]}')
        train_config = BC_TRAIN_CONFIGS[candidate]
        set_seed(int(train_config.seed) + i * 101 + list(WALK_FORWARD_WINDOWS).index(fold) * 1009)
        fit = fit_state_objective_bc_actor_from_frame(
            cache['train_frame'],
            targets=fold_targets,
            profile=profile,
            train_config=train_config,
            sample_weight_config=weight_config,
            sample_size=0,
            rng_seed=int(train_config.seed),
        )
        fit_row = {
            'fold': fold,
            'candidate': candidate,
            'target_rows': int(len(fold_targets)),
            'bc_samples': int(len(fit.actions)),
            'unique_actions': int(pd.Series(fit.actions).nunique()) if len(fit.actions) else 0,
            'train_config': json.dumps(asdict(train_config), sort_keys=True),
            'weight_config': json.dumps(asdict(weight_config), sort_keys=True),
        }
        fit_row.update({k: float(v) if isinstance(v, (int, float, np.number)) and math.isfinite(float(v)) else v for k, v in fit.metrics.items()})
        fit_rows.append(fit_row)

        bc_dataset = build_state_objective_bc_dataset_from_frame(cache['train_frame'], targets=fold_targets, profile=profile, sample_size=0)
        slice_metrics = evaluate_bc_actor_slice_fit(fit.actor, fit.obs, fit.actions, bc_dataset.metadata, slice_cols=SLICE_COLS)
        slice_gate, slice_details, failed_slices = slice_gate_from_metrics(slice_metrics)
        tagged_slices = slice_metrics.copy()
        tagged_slices.insert(0, 'candidate', candidate)
        tagged_slices.insert(0, 'fold', fold)
        slice_frames.append(tagged_slices)

        action_dist = fit.action_distribution.copy()
        action_dist.insert(0, 'candidate', candidate)
        action_dist.insert(0, 'fold', fold)
        action_frames.append(action_dist)

        bt = backtest_profile(
            fit.actor,
            cache['feature_full'],
            cache['price_full'],
            position_cap_config=profile.position_cap,
            position_budget_config=profile.position_budget,
            execution_config=profile.execution,
            mode_router_config=profile.mode_router,
            asset_idx=0,
            feature_columns=feature_columns,
            model_config=profile.model,
            test_seg=cache['warm_seg'],
            trace_decisions=True,
        )
        daily = slice_warm_start_daily_path(
            portfolio_df=bt['portfolio'],
            price_series=cache['price_full'],
            eval_seg=spec['test'],
            position_history=bt.get('position_history', []),
            trade_signals=bt.get('trade_signals', []),
        )
        daily_path = DAILY_DIR / f'{fold}__{candidate}_official_daily_path.csv'
        daily.to_csv(daily_path, index=False)
        metrics = daily_path_metrics(daily)
        drawdown = max_drawdown_interval_metrics(daily)
        drawdown_rows.append({'fold': fold, 'candidate': candidate, **drawdown, 'daily_path': str(daily_path.relative_to(ROOT))})
        daily_manifest.append({
            'fold': fold,
            'candidate': candidate,
            'path': str(daily_path.relative_to(ROOT)),
            'rows': int(len(daily)),
            'start': metrics.get('start'),
            'end': metrics.get('end'),
        })

        runnable_base = runnable_by_fold.get(fold, {})
        drift_score = score_bc_walkforward_drift(
            fold,
            candidate_metrics=metrics,
            baseline_metrics=runnable_base,
            slice_gate=slice_gate,
            target_relative_return_pp=TARGET_RELATIVE_RETURN,
            target_max_dd_pct=TARGET_MAX_DD,
            relative_drift_floor_pp=DRIFT_RELATIVE_FLOOR_PP,
            max_dd_drift_ceiling_pp=DRIFT_MAX_DD_CEILING_PP,
        )
        strict_max_dd_gate = bool(float(metrics['max_dd']) < TARGET_MAX_DD)
        relative_gate = bool(float(metrics['relative_return']) >= TARGET_RELATIVE_RETURN)
        candidate_gate_strict = bool(relative_gate and strict_max_dd_gate and slice_gate and bool(drift_score['drift_gate']))
        row = {
            'fold': fold,
            'candidate': candidate,
            'status': 'evaluated',
            'warm_start': cache['warm_seg'][0],
            'warm_end': cache['warm_seg'][1],
            'target_rows': int(len(fold_targets)),
            'bc_samples': int(len(fit.actions)),
            'return': float(metrics['return']),
            'buy_and_hold_return': float(metrics['buy_and_hold_return']),
            'relative_return': float(metrics['relative_return']),
            'max_dd': float(metrics['max_dd']),
            'avg_pos': float(metrics['avg_pos']),
            'trade_count': int(pd.Series(daily['action']).astype(str).str.upper().isin(['BUY', 'SELL']).sum()),
            'target_relative_return': TARGET_RELATIVE_RETURN,
            'target_max_dd_strict_lt': TARGET_MAX_DD,
            'relative_gate': relative_gate,
            'max_dd_gate_strict': strict_max_dd_gate,
            'slice_gate': bool(slice_gate),
            'failed_slices': ','.join(failed_slices),
            'drift_gate': bool(drift_score['drift_gate']),
            'candidate_gate_strict': candidate_gate_strict,
            'relative_delta_vs_runnable': float(drift_score['relative_delta_vs_baseline']),
            'max_dd_delta_vs_runnable': float(drift_score['max_dd_delta_vs_baseline']),
            'runnable_reference_relative': runnable_base.get('relative_return', np.nan),
            'runnable_reference_max_dd': runnable_base.get('max_dd', np.nan),
            'runnable_reference_avg_pos': runnable_base.get('avg_pos', np.nan),
            'daily_path': str(daily_path.relative_to(ROOT)),
            'slice_details_json': json.dumps(json_safe(slice_details), sort_keys=True),
        }
        row['selection_loss'] = candidate_loss(row)
        candidate_rows.append(row)

        for baseline_name, baseline_by_fold in [('soxlbacktest_legacy_default', soxl_legacy_by_fold), ('soxlbacktest_wf_pass_risk_floor', soxl_risk_floor_by_fold)]:
            baseline = baseline_by_fold.get(fold, {})
            soxl_baseline_rows.append({
                'fold': fold,
                'candidate': candidate,
                'baseline': baseline_name,
                'candidate_relative_return': float(metrics['relative_return']),
                'candidate_max_dd': float(metrics['max_dd']),
                'candidate_avg_pos': float(metrics['avg_pos']),
                'baseline_relative_return': baseline.get('relative_return', np.nan),
                'baseline_max_dd': baseline.get('max_dd', np.nan),
                'baseline_avg_pos': baseline.get('avg_pos', np.nan),
                'relative_delta_vs_baseline': float(metrics['relative_return']) - float(baseline.get('relative_return', np.nan)) if baseline else np.nan,
                'max_dd_delta_vs_baseline': float(metrics['max_dd']) - float(baseline.get('max_dd', np.nan)) if baseline else np.nan,
            })

fit_metrics = pd.DataFrame(fit_rows)
candidate_metrics = pd.DataFrame(candidate_rows)
slice_metrics_all = pd.concat(slice_frames, ignore_index=True) if slice_frames else pd.DataFrame()
action_distribution = pd.concat(action_frames, ignore_index=True) if action_frames else pd.DataFrame()
daily_manifest_df = pd.DataFrame(daily_manifest)
drawdown_intervals = pd.DataFrame(drawdown_rows)
soxl_baseline_comparison = pd.DataFrame(soxl_baseline_rows)

if not candidate_metrics.empty:
    candidate_metrics = candidate_metrics.sort_values(['fold', 'candidate_gate_strict', 'selection_loss', 'candidate'], ascending=[True, False, True, True]).reset_index(drop=True)

fit_metrics.to_csv(CARD / 'bc_fit_metrics.csv', index=False)
candidate_metrics.to_csv(CARD / 'bc_candidate_metrics.csv', index=False)
slice_metrics_all.to_csv(CARD / 'bc_slice_metrics.csv', index=False)
action_distribution.to_csv(CARD / 'bc_action_distribution.csv', index=False)
daily_manifest_df.to_csv(CARD / 'official_daily_path_manifest.csv', index=False)
drawdown_intervals.to_csv(CARD / 'drawdown_interval_diagnostic.csv', index=False)
soxl_baseline_comparison.to_csv(CARD / 'soxlbacktest_baseline_comparison.csv', index=False)

display(candidate_metrics)


[fold:wf_2020] candidate=balanced_sparse targets=2470 warm=('2019-10-07', '2020-12-31')


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


[20293:MainThread](2026-06-15 09:22:02,878) WARNING - qlib.data - [data.py:665] - load calendar error: freq=day, future=True; return current calendar!


[20293:MainThread](2026-06-15 09:22:02,879) WARNING - qlib.data - [data.py:668] - You can get future calendar by referring to the following document: https://github.com/microsoft/qlib/blob/main/scripts/data_collector/contrib/README.md


[20293:MainThread](2026-06-15 09:22:02,882) WARNING - qlib.BaseExecutor - [executor.py:121] - `common_infra` is not set for <qlib.backtest.executor.SimulatorExecutor object at 0x136eb5850>


...

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rzi

,fold,candidate,status,warm_start,warm_end,target_rows,bc_samples,return,buy_and_hold_return,relative_return,...,drift_gate,candidate_gate_strict,relative_delta_vs_runnable,max_dd_delta_vs_runnable,runnable_reference_relative,runnable_reference_max_dd,runnable_reference_avg_pos,daily_path,slice_details_json,selection_loss
0,wf_2020,risk_guarded,evaluated,2019-10-07,2020-12-31,2470,2156,17.509465,58.093571,-40.584106,...,False,False,-62.865745,14.744835,22.281639,21.106156,0.425202,runtime/model_selection/card/release16_nonconv...,"{""drawdown_control_state"": {""accuracy"": 0.7171...",217.343484
1,wf_2020,balanced_sparse,evaluated,2019-10-07,2020-12-31,2470,2156,27.148549,58.093571,-30.945022,...,False,False,-53.226661,17.968766,22.281639,21.106156,0.425202,runtime/model_selection/card/release16_nonconv...,"{""drawdown_control_state"": {""accuracy"": 0.5187...",229.408903
2,wf_2020,release_heavy,evaluated,2019-10-07,2020-12-31,2470,2156,27.148549,58.093571,-30.945022,...,False,False,-53.226661,17.968766,22.281639,21.106156,0.425202,runtime/model_selection/card/release16_nonconv...,"{""drawdown_control_state"": {""accuracy"": 0.5187...",229.408903
3,wf_2022,risk_guarded,evaluated,2021-10-07,2022-12-30,253,252,-9.456403,-86.588072,77.131669,...,False,False,4.680809,2.554115,72.450860,15.072639,0.087837,runtime/model_selection/card/release16_nonconv...,"{""drawdown_control_state"": {""accuracy"": null, ...",13.108229
4,wf_2022,release_heavy,evaluated,2021-10-07,2022-12-30,253,252,-17.467841,-86.588072,69.120231,...,False,False,-3.330629,3.073359,72.450860,15.072639,0.087837,runtime/model_selection/card/release16_nonconv...,"{""drawdown_control_state"": {""accuracy"": null, ...",14.146719
5,wf_2022,balanced_sparse,evaluated,2021-10-07,2022-12-30,253,252,-12.658273,-86.588072,73.929798,...,False,False,1.478938,4.065908,72.450860,15.072639,0.087837,runtime/model_selection/card/release16_nonconv...,"{""drawdown_control_state"": {""accuracy"": null, ...",16.131816
6,wf_2023_2024,release_heavy,evaluated,2023-10-05,2024-12-31,756,755,20.255117,-0.356635,20.611751,...,False,False,6.309689,11.571934,14.302062,13.959434,0.197766,runtime/model_selection/card/release16_nonconv...,"{""drawdown_control_state"": {""accuracy"": null, ...",62.657592
7,wf_2023_2024,balanced_sparse,evaluated,2023-10-05,2024-12-31,756,755,16.140444,-0.356635,16.497078,...,False,False,2.195016,16.834046,14.302062,13.959434,0.197766,runtime/model_selection/card/release16_nonconv...,"{""drawdown_control_state"": {""accuracy"": null, ...",98.344936
8,wf_2023_2024,risk_guarded,evaluated,2023-10-05,2024-12-31,756,755,-6.958390,-0.356635,-6.601755,...,False,False,-20.903818,13.041863,14.302062,13.959434,0.197766,runtime/model_selection/card/release16_nonconv...,"{""drawdown_control_state"": {""accuracy"": null, ...",114.594487


In [5]:

selection_rows: list[dict[str, Any]] = []
selected_daily_paths: list[str] = []
for fold in WALK_FORWARD_WINDOWS:
    fold_candidates = candidate_metrics[candidate_metrics['fold'].astype(str).eq(fold)].copy()
    if fold_candidates.empty:
        selection_rows.append({'fold': fold, 'status': 'missing', 'selected_candidate': None})
        continue
    selected = fold_candidates.sort_values(['candidate_gate_strict', 'selection_loss', 'candidate'], ascending=[False, True, True]).iloc[0].to_dict()
    selection_rows.append({
        'fold': fold,
        'status': 'evaluated',
        'selected_candidate': selected['candidate'],
        'selected_candidate_gate': bool(selected['candidate_gate_strict']),
        'selected_relative_return': float(selected['relative_return']),
        'selected_max_dd': float(selected['max_dd']),
        'selected_avg_pos': float(selected['avg_pos']),
        'selected_trade_count': int(selected['trade_count']),
        'selected_loss': float(selected['selection_loss']),
        'relative_gate': bool(selected['relative_gate']),
        'max_dd_gate_strict': bool(selected['max_dd_gate_strict']),
        'slice_gate': bool(selected['slice_gate']),
        'drift_gate': bool(selected['drift_gate']),
        'failed_slices': selected.get('failed_slices', ''),
        'relative_delta_vs_runnable': float(selected['relative_delta_vs_runnable']),
        'max_dd_delta_vs_runnable': float(selected['max_dd_delta_vs_runnable']),
        'daily_path': selected['daily_path'],
    })
    selected_daily_paths.append(str(selected['daily_path']))

selection_by_fold = pd.DataFrame(selection_rows)
selection_by_fold.to_csv(CARD / 'bc_selection_by_fold.csv', index=False)

# Re-score the selected rows through the shared drift summarizer for a compact gate summary.
drift_scores = []
for _, row in selection_by_fold.iterrows():
    fold = str(row['fold'])
    candidate_metrics_map = {
        'relative_return': row.get('selected_relative_return'),
        'max_dd': row.get('selected_max_dd'),
        'avg_pos': row.get('selected_avg_pos'),
    }
    drift_scores.append(score_bc_walkforward_drift(
        fold,
        candidate_metrics=candidate_metrics_map,
        baseline_metrics=runnable_by_fold.get(fold, {}),
        slice_gate=bool(row.get('slice_gate')),
        target_relative_return_pp=TARGET_RELATIVE_RETURN,
        target_max_dd_pct=TARGET_MAX_DD,
        relative_drift_floor_pp=DRIFT_RELATIVE_FLOOR_PP,
        max_dd_drift_ceiling_pp=DRIFT_MAX_DD_CEILING_PP,
    ))
drift_scores_df = pd.DataFrame(drift_scores)
drift_scores_df.to_csv(CARD / 'bc_drift_scores.csv', index=False)
drift_summary = summarize_bc_walkforward_drift_screen(drift_scores_df, required_folds=tuple(WALK_FORWARD_WINDOWS))

required = list(WALK_FORWARD_WINDOWS)
selected_required = selection_by_fold[selection_by_fold['fold'].isin(required)].copy()
selected_gate_pass_count = int(selected_required['selected_candidate_gate'].fillna(False).astype(bool).sum()) if not selected_required.empty else 0
slice_gate_pass_count = int(selected_required['slice_gate'].fillna(False).astype(bool).sum()) if not selected_required.empty else 0
drift_gate_pass_count = int(selected_required['drift_gate'].fillna(False).astype(bool).sum()) if not selected_required.empty else 0
candidate_gate_pass_count = int(selected_required['relative_gate'].fillna(False).astype(bool).sum()) if not selected_required.empty else 0
strict_dd_pass_count = int(selected_required['max_dd_gate_strict'].fillna(False).astype(bool).sum()) if not selected_required.empty else 0
failed_folds = selected_required.loc[~selected_required['selected_candidate_gate'].fillna(False).astype(bool), 'fold'].astype(str).tolist()
missing_folds = [fold for fold in required if fold not in set(selected_required['fold'].astype(str))]
failed_folds.extend(missing_folds)

if selected_gate_pass_count == len(required):
    decision = 'release16_nonconvex_checkpoint_bc_official_replay_passed_screen_no_actor_retention'
    next_mechanism = 'run_full_fixed_split_and_retention_review_before_actor_retention'
elif slice_gate_pass_count < len(required):
    decision = 'release16_nonconvex_checkpoint_bc_slice_failed_no_promotion'
    next_mechanism = 'repair_sparse_slice_fit_or_state_head_feature_before_ppo'
elif strict_dd_pass_count < len(required):
    decision = 'release16_nonconvex_checkpoint_bc_drawdown_failed_no_promotion'
    next_mechanism = 'add_drawdown_drift_penalty_or_checkpoint_scoring_before_ppo'
elif candidate_gate_pass_count < len(required):
    decision = 'release16_nonconvex_checkpoint_bc_relative_floor_failed_no_promotion'
    next_mechanism = 'revise_participation_target_or_checkpoint_objective_before_ppo'
elif drift_gate_pass_count < len(required):
    decision = 'release16_nonconvex_checkpoint_bc_drift_failed_no_promotion'
    next_mechanism = 'use_bc_as_reference_and_add_drift_stop_before_longer_ppo'
else:
    decision = 'release16_nonconvex_checkpoint_bc_mixed_no_promotion'
    next_mechanism = 'inspect_failed_fold_action_distribution_then_try_drawdown_drift_reward_or_checkpoint_selector'

selected_by_fold = {}
metrics = []
for _, row in selected_required.iterrows():
    fold = str(row['fold'])
    selected_by_fold[fold] = {
        'selected_candidate': str(row['selected_candidate']),
        'selected_candidate_gate': bool(row['selected_candidate_gate']),
        'selected_relative_return': float(row['selected_relative_return']),
        'selected_max_dd': float(row['selected_max_dd']),
        'selected_avg_pos': float(row['selected_avg_pos']),
        'selected_trade_count': int(row['selected_trade_count']),
        'relative_gate': bool(row['relative_gate']),
        'max_dd_gate_strict': bool(row['max_dd_gate_strict']),
        'slice_gate': bool(row['slice_gate']),
        'drift_gate': bool(row['drift_gate']),
        'failed_slices': str(row.get('failed_slices', '')),
        'relative_delta_vs_runnable': float(row['relative_delta_vs_runnable']),
        'max_dd_delta_vs_runnable': float(row['max_dd_delta_vs_runnable']),
    }
    metrics.append({
        'window': fold,
        'split_type': 'walk_forward_test',
        'relative_return_pct': float(row['selected_relative_return']),
        'max_dd_pct': float(row['selected_max_dd']),
        'avg_pos': float(row['selected_avg_pos']),
    })

min_relative = min((float(r['relative_return_pct']) for r in metrics), default=float('nan'))
worst_max_dd = max((float(r['max_dd_pct']) for r in metrics), default=float('nan'))
summary = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'bc_warm_start', 'checkpoint_objective', 'official_replay'],
    'reference_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
    'direct_comparison_baseline': 'release16_nonconvex_checkpoint_objective_score_handoff_v1',
    'source_card': 'release16_nonconvex_checkpoint_objective_score_handoff_v1',
    'source_target_contract': 'release16_failedfold_state_head_target_recipe_v1',
    'source_paths': [
        str((SOURCE_SCORE_CARD / 'summary.json').relative_to(ROOT)),
        str((TARGETS_PATH).relative_to(ROOT)),
        str((RUNNABLE_REFERENCE_PATH).relative_to(ROOT)),
        str((SOXL_BASELINE_METRICS_PATH).relative_to(ROOT)),
    ],
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'ppo_allowed_next': False,
    'bc_screen_allowed_next': False,
    'source_target_contract_gate': True,
    'source_bc_readiness_gate': bool(readiness_summary.get('bc_readiness_gate', False)),
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'drawdown_gate_operator': '<',
    'target_window': required,
    'metrics': metrics,
    'selected_by_fold': selected_by_fold,
    'failed_folds': failed_folds,
    'selected_gate_pass_count': selected_gate_pass_count,
    'candidate_gate_pass_count': candidate_gate_pass_count,
    'strict_dd_pass_count': strict_dd_pass_count,
    'slice_gate_pass_count': slice_gate_pass_count,
    'drift_gate_pass_count': drift_gate_pass_count,
    'min_wf_relative_return_pp': min_relative,
    'worst_wf_max_drawdown_pct': worst_max_dd,
    'max_relative_gap_to_target_pp': max(0.0, TARGET_RELATIVE_RETURN - min_relative) if math.isfinite(min_relative) else None,
    'max_drawdown_gap_to_target_pp': max(0.0, worst_max_dd - TARGET_MAX_DD) if math.isfinite(worst_max_dd) else None,
    'drift_summary': drift_summary,
    'next_mechanism': next_mechanism,
    'primary_blocker': 'bc_candidate_or_slice_gate_failed' if failed_folds else 'diagnostic_no_promotion',
    'training_allowed_from_this_card': False,
    'summary_points': [
        f'Selected BC candidates passed {selected_gate_pass_count}/{len(required)} fold gates under official warm-start replay.',
        f'Slice gate passed {slice_gate_pass_count}/{len(required)} folds; drift gate passed {drift_gate_pass_count}/{len(required)} folds.',
        f'Min selected relative return is {min_relative:.2f}pp and worst selected MaxDD is {worst_max_dd:.2f}%.',
        'SOXLBacktest qlib baseline comparison is written as soxlbacktest_baseline_comparison.csv.',
        'No actor is retained from this diagnostic screen.',
    ],
    'sidecars': {
        'target_coverage': str((CARD / 'target_coverage.csv').relative_to(ROOT)),
        'bc_fit_metrics': str((CARD / 'bc_fit_metrics.csv').relative_to(ROOT)),
        'bc_candidate_metrics': str((CARD / 'bc_candidate_metrics.csv').relative_to(ROOT)),
        'bc_selection_by_fold': str((CARD / 'bc_selection_by_fold.csv').relative_to(ROOT)),
        'bc_slice_metrics': str((CARD / 'bc_slice_metrics.csv').relative_to(ROOT)),
        'bc_action_distribution': str((CARD / 'bc_action_distribution.csv').relative_to(ROOT)),
        'bc_drift_scores': str((CARD / 'bc_drift_scores.csv').relative_to(ROOT)),
        'drawdown_interval_diagnostic': str((CARD / 'drawdown_interval_diagnostic.csv').relative_to(ROOT)),
        'official_daily_path_manifest': str((CARD / 'official_daily_path_manifest.csv').relative_to(ROOT)),
        'soxlbacktest_baseline_comparison': str((CARD / 'soxlbacktest_baseline_comparison.csv').relative_to(ROOT)),
        'selection': str((CARD / 'selection.json').relative_to(ROOT)),
        'summary': str((CARD / 'summary.json').relative_to(ROOT)),
        'conclusion': str((CARD / 'conclusion.md').relative_to(ROOT)),
    },
}
selection = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
    'source_card': 'release16_nonconvex_checkpoint_objective_score_handoff_v1',
    'direct_comparison_baseline': 'release16_nonconvex_checkpoint_objective_score_handoff_v1',
    'reference_baseline': 'soxlbacktest_qlib_walkforward_baseline_v1',
    'selected_by_fold': selected_by_fold,
    'next_mechanism': next_mechanism,
    'summary': {
        'selected_gate_pass_count': selected_gate_pass_count,
        'failed_folds': failed_folds,
        'min_wf_relative_return_pp': min_relative,
        'worst_wf_max_drawdown_pct': worst_max_dd,
    },
}
write_json(CARD / 'summary.json', summary)
write_json(CARD / 'selection.json', selection)

conclusion = (
    f'# {EXPERIMENT}\n\n'
    f'Decision: `{decision}`.\n\n'
    f'Selected BC official replay passes `{selected_gate_pass_count}/{len(required)}` required fold gates. '
    f'Min selected relative return is `{min_relative:.2f}pp`; worst selected MaxDD is `{worst_max_dd:.2f}%`; '
    f'slice gate passes `{slice_gate_pass_count}/{len(required)}` folds; drift gate passes `{drift_gate_pass_count}/{len(required)}` folds.\n\n'
    f'No actor is retained. Next mechanism: `{next_mechanism}`.\n\n'
    'Primary sidecars: `bc_candidate_metrics.csv`, `bc_selection_by_fold.csv`, `bc_slice_metrics.csv`, '
    '`bc_drift_scores.csv`, `soxlbacktest_baseline_comparison.csv`, and `official_daily_paths/`.\n'
)
(CARD / 'conclusion.md').write_text(conclusion, encoding='utf-8')

display(selection_by_fold)
display(drift_scores_df)
Markdown(conclusion)


,fold,status,selected_candidate,selected_candidate_gate,selected_relative_return,selected_max_dd,selected_avg_pos,selected_trade_count,selected_loss,relative_gate,max_dd_gate_strict,slice_gate,drift_gate,failed_slices,relative_delta_vs_runnable,max_dd_delta_vs_runnable,daily_path
0,wf_2020,evaluated,risk_guarded,False,-40.584106,35.850991,0.403334,39,217.343484,False,False,True,False,,-62.865745,14.744835,runtime/model_selection/card/release16_nonconv...
1,wf_2022,evaluated,risk_guarded,False,77.131669,17.626753,0.083748,32,13.108229,True,True,False,False,"release_gate_state,is_rebound_cluster,drawdown...",4.680809,2.554115,runtime/model_selection/card/release16_nonconv...
2,wf_2023_2024,evaluated,release_heavy,False,20.611751,25.531369,0.353351,37,62.657592,False,False,False,False,"release_gate_state,is_downtrend_state,drawdown...",6.309689,11.571934,runtime/model_selection/card/release16_nonconv...


,fold,candidate_relative_return,candidate_max_dd,candidate_avg_pos,baseline_relative_return,baseline_max_dd,baseline_avg_pos,relative_delta_vs_baseline,max_dd_delta_vs_baseline,candidate_gate,baseline_gate,slice_gate,drift_gate,fold_gate,target_relative_return_pp,target_max_dd_pct,relative_drift_floor_pp,max_dd_drift_ceiling_pp,failed_checks
0,wf_2020,-40.584106,35.850991,0.403334,22.281639,21.106156,0.425202,-62.865745,14.744835,False,False,True,False,False,30.0,20.0,-5.0,2.0,"[candidate_relative_return, candidate_max_dd, ..."
1,wf_2022,77.131669,17.626753,0.083748,72.450860,15.072639,0.087837,4.680809,2.554115,True,True,False,False,False,30.0,20.0,-5.0,2.0,"[slice_gate, max_dd_drift]"
2,wf_2023_2024,20.611751,25.531369,0.353351,14.302062,13.959434,0.197766,6.309689,11.571934,False,False,False,False,False,30.0,20.0,-5.0,2.0,"[candidate_relative_return, candidate_max_dd, ..."


# release16_nonconvex_checkpoint_bc_official_replay_v1

Decision: `release16_nonconvex_checkpoint_bc_slice_failed_no_promotion`.

Selected BC official replay passes `0/3` required fold gates. Min selected relative return is `-40.58pp`; worst selected MaxDD is `35.85%`; slice gate passes `1/3` folds; drift gate passes `0/3` folds.

No actor is retained. Next mechanism: `repair_sparse_slice_fit_or_state_head_feature_before_ppo`.

Primary sidecars: `bc_candidate_metrics.csv`, `bc_selection_by_fold.csv`, `bc_slice_metrics.csv`, `bc_drift_scores.csv`, `soxlbacktest_baseline_comparison.csv`, and `official_daily_paths/`.


## Conclusion

The executed cells above are the source of truth. This notebook intentionally keeps `promotion_allowed=false` and `actor_retention_allowed=false`; a retained actor still requires complete fixed-split / walk-forward replay evidence after any candidate passes this BC official replay screen.
